# YouTube Comment & Channel Metadata Scraper v2
**Project:** SpamShield — Deteksi Spam Promosi Judi Online pada Komentar YouTube Indonesia  
**Authors:** Ricky (2702243016), Elmer Williams, Nicholas Sinclair Alfianto  
**Institution:** Bina Nusantara University  

---

### Changelog from v1
| Change | Reason |
|--------|--------|
| Added `source_channel` tracking | Analisis distribusi spam per genre channel |
| Added checkpointing (auto-save per video) | Resume jika quota habis tanpa kehilangan data |
| Added full reply fetching via `comments().list()` | v1 hanya ambil max 5 replies per thread |
| Added audit log with timestamps | Dokumentasi akademik: kapan, berapa, status |
| Added data integrity validation | Memastikan kolom konsisten sebelum export |
| Added `scrape_date` column | Reproducibility — kapan data diambil |

### Data Collection Methodology
- **API:** YouTube Data API v3  
- **Text Format:** plainText (stripped HTML, no formatting)  
- **Scope:** All top-level comments + ALL replies (not limited to 5 per thread)  
- **Channel Metadata:** Batch-fetched (50 channels/request) for efficiency  
- **Deduplication:** By video_id at load time, by comment_id at export  

## 1. Setup

In [ ]:
%pip install google-api-python-client python-dotenv tqdm --quiet

In [1]:
import os
import re
import json
import time
import logging
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.notebook import tqdm

print("Libraries imported.")

Libraries imported.


In [2]:
# Load API key
load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')

if not API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY tidak ditemukan!\n"
        "Buat file .env di folder ini, isi: YOUTUBE_API_KEY=your_key_here"
    )

print(f"API Key loaded: {API_KEY[:8]}...{API_KEY[-4:]}")

API Key loaded: AIzaSyDr...xxYY


In [ ]:
# === KONFIGURASI ===
# Ganti nama output file di sini
OUTPUT_FILENAME = 'dataset_spamshield_batch2'

CONFIG = {
    'output_dir': Path('output'),
    'checkpoint_dir': Path('output/checkpoints'),
    'log_dir': Path('output/logs'),
    'request_delay': 0.3,
    'video_delay': 1.0,
    'max_results_per_page': 100,
    'channel_batch_size': 50,
    'fetch_all_replies': True,  # True = fetch semua replies, False = hanya 5 per thread
}

for d in [CONFIG['output_dir'], CONFIG['checkpoint_dir'], CONFIG['log_dir']]:
    d.mkdir(parents=True, exist_ok=True)

youtube = build('youtube', 'v3', developerKey=API_KEY)

# Setup logging
log_file = CONFIG['log_dir'] / f'scrape_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

SCRAPE_TIMESTAMP = datetime.now(timezone.utc).isoformat()

logger.info(f"Output: {OUTPUT_FILENAME}")
logger.info(f"Scrape timestamp: {SCRAPE_TIMESTAMP}")
logger.info(f"Config: {json.dumps({k: str(v) for k,v in CONFIG.items()}, indent=2)}")
print(f"Output: {OUTPUT_FILENAME}.csv")
print(f"Log: {log_file}")

2026-02-09 17:25:27,011 | INFO | Output: dataset_spamshield
2026-02-09 17:25:27,011 | INFO | Scrape timestamp: 2026-02-09T10:25:27.011119+00:00
2026-02-09 17:25:27,011 | INFO | Config: {
  "output_dir": "output",
  "checkpoint_dir": "output\\checkpoints",
  "log_dir": "output\\logs",
  "request_delay": "0.3",
  "video_delay": "1.0",
  "max_results_per_page": "100",
  "channel_batch_size": "50",
  "fetch_all_replies": "True"
}


Output: dataset_spamshield.csv
Log: output\logs\scrape_20260209_172527.log


## 2. Helper Functions

In [4]:
def extract_video_id(url: str) -> str | None:
    """Extract video ID (11 char) dari berbagai format YouTube URL."""
    patterns = [
        r'(?:v=)([0-9A-Za-z_-]{11})',
        r'(?:youtu\.be\/)([0-9A-Za-z_-]{11})',
        r'(?:embed\/)([0-9A-Za-z_-]{11})',
        r'(?:shorts\/)([0-9A-Za-z_-]{11})',
    ]
    for pattern in patterns:
        match = re.search(pattern, url.strip())
        if match:
            return match.group(1)
    return None


def load_video_urls(filepath: str) -> list[dict]:
    """
    Load video URLs dari file text.
    
    Format file:
    ```
    # [SOURCE_CHANNEL] Channel Name
    https://youtube.com/watch?v=xxx
    https://youtube.com/watch?v=yyy
    
    # [SOURCE_CHANNEL] Another Channel
    https://youtube.com/watch?v=zzz
    ```
    
    Lines starting with '# [TAG]' set the source_channel for subsequent URLs.
    """
    videos = []
    seen_ids = set()
    current_source = 'unknown'

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Parse source channel tag: # [MasterChef] or # [PUBG] etc
            source_match = re.match(r'^#\s*\[(.+?)\]', line)
            if source_match:
                current_source = source_match.group(1).strip()
                continue
            
            if line.startswith('#'):
                continue

            video_id = extract_video_id(line)
            if video_id and video_id not in seen_ids:
                videos.append({
                    'url': line,
                    'video_id': video_id,
                    'source_channel': current_source,
                })
                seen_ids.add(video_id)
            elif video_id in seen_ids:
                print(f"  Line {line_num}: Duplicate, skipped.")
            else:
                print(f"  Line {line_num}: Invalid URL, skipped.")

    return videos


# Quick test
test = "https://www.youtube.com/watch?v=u-zEbT8q-GQ"
print(f"Test: {test} -> {extract_video_id(test)}")

Test: https://www.youtube.com/watch?v=u-zEbT8q-GQ -> u-zEbT8q-GQ


## 3. API Functions

In [5]:
def get_video_metadata(video_id: str) -> dict | None:
    """Ambil metadata video: judul, views, likes, jumlah komentar."""
    try:
        response = youtube.videos().list(
            part='snippet,statistics',
            id=video_id
        ).execute()

        if not response.get('items'):
            logger.warning(f"Video not found: {video_id}")
            return None

        item = response['items'][0]
        snippet = item['snippet']
        stats = item['statistics']

        return {
            'video_id': video_id,
            'video_title': snippet.get('title', ''),
            'video_channel_id': snippet.get('channelId', ''),
            'video_channel_title': snippet.get('channelTitle', ''),
            'video_category_id': snippet.get('categoryId', ''),
            'video_published_at': snippet.get('publishedAt', ''),
            'video_view_count': int(stats.get('viewCount', 0)),
            'video_like_count': int(stats.get('likeCount', 0)),
            'video_comment_count': int(stats.get('commentCount', 0)),
        }

    except HttpError as e:
        logger.error(f"Video API error {video_id}: {e.resp.status}")
        return None

In [6]:
def get_channel_metadata_batch(channel_ids: list[str]) -> dict[str, dict]:
    """
    Batch-fetch channel metadata (max 50 per request).
    50 channels / 1 request = 1 quota unit (vs 50 units jika satu-satu).
    """
    DEFAULT = {
        'channel_subscriber_count': 0,
        'channel_video_count': 0,
        'channel_view_count': 0,
        'channel_created_at': '',
        'channel_description_length': 0,
        'channel_hidden_subscriber': False,
    }

    results = {}
    batch_size = CONFIG['channel_batch_size']

    for i in range(0, len(channel_ids), batch_size):
        batch = channel_ids[i:i + batch_size]

        try:
            response = youtube.channels().list(
                part='snippet,statistics',
                id=','.join(batch)
            ).execute()

            for item in response.get('items', []):
                ch_id = item['id']
                snippet = item.get('snippet', {})
                stats = item.get('statistics', {})
                hidden_subs = stats.get('hiddenSubscriberCount', False)

                results[ch_id] = {
                    'channel_subscriber_count': 0 if hidden_subs else int(stats.get('subscriberCount', 0)),
                    'channel_video_count': int(stats.get('videoCount', 0)),
                    'channel_view_count': int(stats.get('viewCount', 0)),
                    'channel_created_at': snippet.get('publishedAt', ''),
                    'channel_description_length': len(snippet.get('description', '')),
                    'channel_hidden_subscriber': hidden_subs,
                }

            time.sleep(CONFIG['request_delay'])

        except HttpError as e:
            logger.error(f"Channel batch error: {e.resp.status}")

    # Default untuk channel yang gagal
    for ch_id in channel_ids:
        if ch_id not in results:
            results[ch_id] = DEFAULT.copy()

    return results

In [7]:
def get_all_replies(parent_id: str, video_id: str) -> list[dict]:
    """
    Fetch ALL replies untuk satu comment thread.
    
    commentThreads().list() hanya return max 5 replies per thread.
    Untuk mendapatkan semua, kita perlu pakai comments().list() dengan parentId.
    """
    replies = []
    next_page_token = None

    while True:
        try:
            response = youtube.comments().list(
                part='snippet',
                parentId=parent_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat='plainText'
            ).execute()

            for item in response.get('items', []):
                rs = item['snippet']
                replies.append({
                    'comment_id': item['id'],
                    'parent_id': parent_id,
                    'is_reply': True,
                    'comment_text': rs.get('textDisplay', ''),
                    'comment_published_at': rs.get('publishedAt', ''),
                    'comment_updated_at': rs.get('updatedAt', ''),
                    'author_channel_id': rs.get('authorChannelId', {}).get('value', ''),
                    'author_display_name': rs.get('authorDisplayName', ''),
                    'like_count': int(rs.get('likeCount', 0)),
                    'reply_count': 0,
                })

            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
            time.sleep(CONFIG['request_delay'])

        except HttpError as e:
            logger.error(f"Reply fetch error for {parent_id}: {e.resp.status}")
            break

    return replies

In [8]:
def get_video_comments(video_id: str, video_metadata: dict, source_channel: str) -> list[dict]:
    """
    Ambil SEMUA komentar + replies dari video.
    
    Strategy: 3-phase approach
    1. Ambil semua top-level comments (paginated)
    2. Untuk thread dengan >5 replies, fetch ALL replies via comments().list()
    3. Batch-fetch channel metadata untuk semua unique authors
    4. Gabungkan semua
    """
    # ===== PHASE 1: Ambil semua top-level comments =====
    raw_comments = []
    threads_needing_full_replies = []  # (parent_id, reply_count_from_api)
    author_ids = set()
    next_page_token = None

    while True:
        try:
            response = youtube.commentThreads().list(
                part='snippet,replies',
                videoId=video_id,
                maxResults=CONFIG['max_results_per_page'],
                pageToken=next_page_token,
                textFormat='plainText'
            ).execute()

            for item in response.get('items', []):
                # --- Top-level comment ---
                top = item['snippet']['topLevelComment']
                ts = top['snippet']
                author_ch_id = ts.get('authorChannelId', {}).get('value', '')
                total_replies = int(item['snippet'].get('totalReplyCount', 0))

                if author_ch_id:
                    author_ids.add(author_ch_id)

                raw_comments.append({
                    'comment_id': top['id'],
                    'parent_id': None,
                    'is_reply': False,
                    'comment_text': ts.get('textDisplay', ''),
                    'comment_published_at': ts.get('publishedAt', ''),
                    'comment_updated_at': ts.get('updatedAt', ''),
                    'author_channel_id': author_ch_id,
                    'author_display_name': ts.get('authorDisplayName', ''),
                    'like_count': int(ts.get('likeCount', 0)),
                    'reply_count': total_replies,
                })

                # --- Replies handling ---
                inline_replies = item.get('replies', {}).get('comments', [])

                if CONFIG['fetch_all_replies'] and total_replies > len(inline_replies):
                    # Thread punya lebih banyak replies daripada yang di-return inline
                    # Perlu fetch terpisah
                    threads_needing_full_replies.append(top['id'])
                else:
                    # Inline replies sudah cukup (<=5 replies)
                    for reply in inline_replies:
                        rs = reply['snippet']
                        reply_ch_id = rs.get('authorChannelId', {}).get('value', '')
                        if reply_ch_id:
                            author_ids.add(reply_ch_id)

                        raw_comments.append({
                            'comment_id': reply['id'],
                            'parent_id': top['id'],
                            'is_reply': True,
                            'comment_text': rs.get('textDisplay', ''),
                            'comment_published_at': rs.get('publishedAt', ''),
                            'comment_updated_at': rs.get('updatedAt', ''),
                            'author_channel_id': reply_ch_id,
                            'author_display_name': rs.get('authorDisplayName', ''),
                            'like_count': int(rs.get('likeCount', 0)),
                            'reply_count': 0,
                        })

            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
            time.sleep(CONFIG['request_delay'])

        except HttpError as e:
            if e.resp.status == 403:
                if 'commentsDisabled' in str(e):
                    logger.warning(f"Comments disabled: {video_id}")
                elif 'quotaExceeded' in str(e):
                    logger.error("Quota exceeded!")
                    raise
                else:
                    logger.warning(f"Forbidden: {video_id}")
            else:
                logger.error(f"API Error: {e.resp.status}")
            break

    # ===== PHASE 1.5: Fetch full replies for threads that need it =====
    if threads_needing_full_replies:
        tqdm.write(f"    -> Fetching full replies for {len(threads_needing_full_replies)} threads...")
        for parent_id in threads_needing_full_replies:
            full_replies = get_all_replies(parent_id, video_id)
            for r in full_replies:
                if r['author_channel_id']:
                    author_ids.add(r['author_channel_id'])
            raw_comments.extend(full_replies)

    if not raw_comments:
        return []

    # ===== PHASE 2: Batch-fetch channel metadata =====
    unique_authors = list(author_ids)
    tqdm.write(f"    -> {len(raw_comments)} comments, {len(unique_authors)} unique authors")

    channel_meta_map = get_channel_metadata_batch(unique_authors)

    # ===== PHASE 3: Gabungkan semua =====
    DEFAULT_CH = {
        'channel_subscriber_count': 0,
        'channel_video_count': 0,
        'channel_view_count': 0,
        'channel_created_at': '',
        'channel_description_length': 0,
        'channel_hidden_subscriber': False,
    }

    final = []
    for c in raw_comments:
        ch_meta = channel_meta_map.get(c['author_channel_id'], DEFAULT_CH)
        final.append({
            **c,
            **ch_meta,
            **video_metadata,
            'source_channel': source_channel,
            'scrape_date': SCRAPE_TIMESTAMP,
        })

    return final

## 4. Main Scraper (with Checkpointing)

In [9]:
def load_checkpoint() -> set:
    """Load video IDs yang sudah berhasil di-scrape dari checkpoint files."""
    done = set()
    for f in CONFIG['checkpoint_dir'].glob('*.csv'):
        try:
            df = pd.read_csv(f, usecols=['video_id'], encoding='utf-8-sig')
            done.update(df['video_id'].unique())
        except Exception:
            pass
    return done


def save_checkpoint(comments: list[dict], video_id: str):
    """Save komentar satu video sebagai checkpoint file."""
    if not comments:
        return
    filepath = CONFIG['checkpoint_dir'] / f'cp_{video_id}.csv'
    pd.DataFrame(comments).to_csv(filepath, index=False, encoding='utf-8-sig')


def scrape_videos(video_list: list[dict], resume: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Scrape semua video dalam list.
    
    - resume=True: skip video yang sudah ada di checkpoint
    - Auto-save setiap video selesai
    - Quota exceeded -> save progress, bisa dilanjut nanti
    """
    already_done = load_checkpoint() if resume else set()
    if already_done:
        logger.info(f"Resuming: {len(already_done)} videos already scraped")

    all_comments = []
    video_stats = []

    for idx, video in enumerate(tqdm(video_list, desc="Scraping")):
        video_id = video['video_id']
        source_channel = video.get('source_channel', 'unknown')

        # Skip if already done
        if video_id in already_done:
            tqdm.write(f"  [SKIP] {video_id} (already scraped)")
            continue

        # Step 1: Video metadata
        metadata = get_video_metadata(video_id)
        if not metadata:
            video_stats.append({
                'video_id': video_id, 'source_channel': source_channel,
                'status': 'error_metadata', 'comments_scraped': 0
            })
            continue

        # Step 2: Comments + channel metadata
        try:
            comments = get_video_comments(video_id, metadata, source_channel)

            # Checkpoint: save per video
            save_checkpoint(comments, video_id)

            video_stats.append({
                'video_id': video_id,
                'video_title': metadata['video_title'],
                'video_channel_title': metadata['video_channel_title'],
                'source_channel': source_channel,
                'video_comment_count_api': metadata['video_comment_count'],
                'comments_scraped': len(comments),
                'status': 'success'
            })

            all_comments.extend(comments)
            logger.info(f"[{idx+1}/{len(video_list)}] {metadata['video_title'][:50]}... -> {len(comments)} comments")
            tqdm.write(f"  [{idx+1}/{len(video_list)}] {metadata['video_title'][:50]}... -> {len(comments)}")

        except HttpError as e:
            if 'quotaExceeded' in str(e):
                logger.error("QUOTA EXCEEDED! Saving progress. Run lagi besok untuk lanjut.")
                print("\nQuota exceeded! Progress saved. Run lagi besok untuk lanjut.")
                break
            video_stats.append({
                'video_id': video_id, 'source_channel': source_channel,
                'status': 'error_comments', 'comments_scraped': 0
            })

        time.sleep(CONFIG['video_delay'])

    # Jika resume, merge dengan checkpoint data
    if resume and already_done:
        logger.info("Merging with checkpoint data...")
        for f in CONFIG['checkpoint_dir'].glob('*.csv'):
            try:
                cp_df = pd.read_csv(f, encoding='utf-8-sig')
                all_comments.extend(cp_df.to_dict('records'))
            except Exception:
                pass

    return pd.DataFrame(all_comments), pd.DataFrame(video_stats)

## 5. Load URLs & Run

In [10]:
# Load video URLs
# Format file video_urls.txt:
#
# # [MasterChef] MasterChef Indonesia
# https://youtube.com/watch?v=xxx
# https://youtube.com/watch?v=yyy
#
# # [PUBG] PUBG Mobile Indonesia
# https://youtube.com/watch?v=zzz

VIDEO_URL_FILE = 'video_urls.txt'

if not Path(VIDEO_URL_FILE).exists():
    raise FileNotFoundError(
        f"File '{VIDEO_URL_FILE}' tidak ditemukan!\n"
        f"Buat file tersebut, isi dengan URL YouTube (satu per baris)."
    )

video_list = load_video_urls(VIDEO_URL_FILE)
print(f"Loaded {len(video_list)} unique videos")

# Show per-source summary
from collections import Counter
source_counts = Counter(v['source_channel'] for v in video_list)
print(f"\nPer source:")
for source, count in source_counts.most_common():
    print(f"  [{source}]: {count} videos")

Loaded 106 unique videos

Per source:
  [XIN]: 25 videos
  [ML]: 21 videos
  [JebreetMedia]: 19 videos
  [MasterChef]: 14 videos
  [PUBG]: 10 videos
  [FrostDiamond]: 10 videos
  [FerryIrwandi]: 6 videos
  [RaymondChin]: 1 videos


In [11]:
# === RUN SCRAPER ===
print(f"Starting scraper for {len(video_list)} videos...")
print("=" * 60)

start_time = time.time()
comments_df, stats_df = scrape_videos(video_list, resume=True)
elapsed = time.time() - start_time

print("=" * 60)
print(f"Done in {elapsed:.1f}s ({elapsed/60:.1f} min)")

Starting scraper for 106 videos...


Scraping:   0%|          | 0/106 [00:00<?, ?it/s]

    -> Fetching full replies for 2 threads...
    -> 95 comments, 85 unique authors


2026-02-09 17:25:55,320 | INFO | [1/106] Hero Spotlight | Kimmy | Hoverjet Outrider | Mobil... -> 95 comments


  [1/106] Hero Spotlight | Kimmy | Hoverjet Outrider | Mobil... -> 95
    -> Fetching full replies for 1 threads...
    -> 88 comments, 75 unique authors


...

  [104/106] 2 0 SAJA BOSSS !!! EVOS HOLY VS HOME BOIS MATCH 2 ... -> 57
    -> 76 comments, 71 unique authors


2026-02-09 17:40:46,622 | INFO | [105/106] GACOR RETRI ALBERTTT BISA BUAT COMEBACK !!! EVOS H... -> 76 comments


  [105/106] GACOR RETRI ALBERTTT BISA BUAT COMEBACK !!! EVOS H... -> 76
    -> Fetching full replies for 1 threads...
    -> 104 comments, 94 unique authors


2026-02-09 17:40:48,815 | INFO | [106/106] THIS IS SUTSUJIN BROKKK MANIAC !!! RRQ HOSHI VS TE... -> 104 comments


  [106/106] THIS IS SUTSUJIN BROKKK MANIAC !!! RRQ HOSHI VS TE... -> 104
Done in 896.3s (14.9 min)


## 6. Data Validation & Export

In [12]:
# === DATA VALIDATION ===
print("=" * 60)
print("DATA VALIDATION")
print("=" * 60)

# Expected columns (the contract for downstream scripts)
EXPECTED_COLUMNS = [
    'comment_id', 'parent_id', 'is_reply', 'comment_text',
    'comment_published_at', 'comment_updated_at',
    'author_channel_id', 'author_display_name',
    'like_count', 'reply_count',
    'channel_subscriber_count', 'channel_video_count',
    'channel_view_count', 'channel_created_at',
    'channel_description_length', 'channel_hidden_subscriber',
    'video_id', 'video_title', 'video_channel_id',
    'video_channel_title', 'video_category_id', 'video_published_at',
    'video_view_count', 'video_like_count', 'video_comment_count',
    'source_channel', 'scrape_date',
]

if len(comments_df) > 0:
    # Check columns
    missing = set(EXPECTED_COLUMNS) - set(comments_df.columns)
    extra = set(comments_df.columns) - set(EXPECTED_COLUMNS)
    
    if missing:
        print(f"  MISSING columns: {missing}")
        logger.error(f"Missing columns: {missing}")
    if extra:
        print(f"  EXTRA columns (will keep): {extra}")
    if not missing:
        print(f"  Column check: PASSED ({len(EXPECTED_COLUMNS)} expected columns present)")

    # Deduplicate by comment_id
    before_dedup = len(comments_df)
    comments_df = comments_df.drop_duplicates(subset=['comment_id'], keep='first')
    after_dedup = len(comments_df)
    print(f"  Dedup comment_id: {before_dedup} -> {after_dedup} (removed {before_dedup - after_dedup})")

    # Null check
    null_text = comments_df['comment_text'].isna().sum()
    null_author = comments_df['author_channel_id'].isna().sum() + (comments_df['author_channel_id'] == '').sum()
    print(f"  Null comment_text: {null_text}")
    print(f"  Empty author_channel_id: {null_author}")

    # Reorder columns
    final_cols = [c for c in EXPECTED_COLUMNS if c in comments_df.columns]
    extra_cols = [c for c in comments_df.columns if c not in EXPECTED_COLUMNS]
    comments_df = comments_df[final_cols + extra_cols]

    print(f"  Final shape: {comments_df.shape}")
    logger.info(f"Validation passed. Shape: {comments_df.shape}")
else:
    print("  No data to validate.")

2026-02-09 17:49:10,590 | INFO | Validation passed. Shape: (57991, 27)


DATA VALIDATION
  Column check: PASSED (27 expected columns present)
  Dedup comment_id: 59165 -> 57991 (removed 1174)
  Null comment_text: 0
  Empty author_channel_id: 0
  Final shape: (57991, 27)


In [13]:
# === EXPORT ===
comments_file = CONFIG['output_dir'] / f'{OUTPUT_FILENAME}.csv'
stats_file = CONFIG['output_dir'] / f'{OUTPUT_FILENAME}_video_stats.csv'

if len(comments_df) > 0:
    comments_df.to_csv(comments_file, index=False, encoding='utf-8-sig')
    size_mb = comments_file.stat().st_size / 1024 / 1024
    print(f"Dataset saved: {comments_file}")
    print(f"  Rows: {len(comments_df):,} | Size: {size_mb:.2f} MB")
    logger.info(f"Exported {len(comments_df)} comments to {comments_file}")
else:
    print("No comments collected.")

stats_df.to_csv(stats_file, index=False, encoding='utf-8-sig')
print(f"Video stats saved: {stats_file}")

2026-02-09 17:49:13,375 | INFO | Exported 57991 comments to output\dataset_spamshield.csv


Dataset saved: output\dataset_spamshield.csv
  Rows: 57,991 | Size: 26.33 MB
Video stats saved: output\dataset_spamshield_video_stats.csv


## 7. Summary & Audit Report

In [14]:
# === SUMMARY ===
print("=" * 60)
print("SCRAPING SUMMARY")
print("=" * 60)

total = len(stats_df)
success = len(stats_df[stats_df['status'] == 'success']) if 'status' in stats_df.columns else 0
print(f"\nVideos: {success}/{total} successful")

if len(comments_df) > 0:
    print(f"\nComments:")
    print(f"  Total           : {len(comments_df):,}")
    print(f"  Top-level       : {len(comments_df[~comments_df['is_reply']]):,}")
    print(f"  Replies         : {len(comments_df[comments_df['is_reply']]):,}")
    print(f"  Unique authors  : {comments_df['author_channel_id'].nunique():,}")
    print(f"  Unique videos   : {comments_df['video_id'].nunique():,}")

    # Per-source breakdown
    if 'source_channel' in comments_df.columns:
        print(f"\nPer source channel:")
        source_summary = comments_df.groupby('source_channel').agg(
            comments=('comment_id', 'count'),
            videos=('video_id', 'nunique'),
            unique_authors=('author_channel_id', 'nunique'),
        ).sort_values('comments', ascending=False)
        for source, row in source_summary.iterrows():
            print(f"  [{source}]: {row['comments']:,} comments, {row['videos']} videos, {row['unique_authors']:,} authors")

    # Quality
    empty = comments_df['comment_text'].isna().sum() + (comments_df['comment_text'] == '').sum()
    dupes = comments_df['comment_id'].duplicated().sum()
    print(f"\nQuality:")
    print(f"  Empty comments  : {empty}")
    print(f"  Duplicate IDs   : {dupes}")

    print(f"\nColumns ({len(comments_df.columns)}):")
    for col in comments_df.columns:
        print(f"  {col}")

logger.info("=== SCRAPING COMPLETE ===")
logger.info(f"Total comments: {len(comments_df)}")
logger.info(f"Videos success: {success}/{total}")

2026-02-09 17:49:15,550 | INFO | === SCRAPING COMPLETE ===
2026-02-09 17:49:15,550 | INFO | Total comments: 57991
2026-02-09 17:49:15,550 | INFO | Videos success: 106/106


SCRAPING SUMMARY

Videos: 106/106 successful

Comments:
  Total           : 57,991
  Top-level       : 43,160
  Replies         : 14,831
  Unique authors  : 39,652
  Unique videos   : 106

Per source channel:
  [FerryIrwandi]: 23,983 comments, 6 videos, 18,960 authors
  [MasterChef]: 20,843 comments, 14 videos, 11,139 authors
  [FrostDiamond]: 5,984 comments, 10 videos, 4,707 authors
  [JebreetMedia]: 2,522 comments, 19 videos, 1,940 authors
  [ML]: 1,747 comments, 21 videos, 1,246 authors
  [XIN]: 1,313 comments, 25 videos, 1,186 authors
  [RaymondChin]: 933 comments, 1 videos, 824 authors
  [PUBG]: 666 comments, 10 videos, 555 authors

Quality:
  Empty comments  : 11
  Duplicate IDs   : 0

Columns (27):
  comment_id
  parent_id
  is_reply
  comment_text
  comment_published_at
  comment_updated_at
  author_channel_id
  author_display_name
  like_count
  reply_count
  channel_subscriber_count
  channel_video_count
  channel_view_count
  channel_created_at
  channel_description_length
 

In [15]:
# === PREVIEW DATA ===
if len(comments_df) > 0:
    print("Sample: Comment text")
    display(comments_df[[
        'comment_text', 'author_display_name',
        'like_count', 'is_reply', 'source_channel',
    ]].head(10))

    print("\nSample: Channel metadata")
    display(comments_df[[
        'author_display_name',
        'channel_subscriber_count', 'channel_video_count',
        'channel_view_count', 'channel_created_at',
        'channel_description_length', 'channel_hidden_subscriber',
    ]].drop_duplicates(subset='author_display_name').head(10))

Sample: Comment text


,comment_text,author_display_name,like_count,is_reply,source_channel
0,Assalamualaikum bang tolong ijin nin pengguna ...,@Shiro-danji,0,False,ML
1,re almeng Suramadu olanya itunya,@DaffaaPR,2,False,ML
2,Jaringan mu ngelag,@adamardiansyah5761,0,False,ML
3,Min kok skin starwars gw kok ilang?,@boys.252,0,False,ML
4,"Baca patch bro, jgn minim literasi",@OnlyRiff,0,True,ML
5,@OnlyRiff gw blm download data patch nya aja ...,@boys.252,0,True,ML
6,Buat moonton pliss lah Hero gue jangan di ruba...,@raysahetapy6474,1,False,ML
7,Tinggal konten gloo revaml,@Awn98,0,False,ML
8,Skil satu nya jadi bagus,@HalimahElin-ob4lf,0,False,ML
9,"ini baru cewe , daripada sebelum di revamp kay...",@dreammashup2214,1,False,ML



Sample: Channel metadata


,author_display_name,channel_subscriber_count,channel_video_count,channel_view_count,channel_created_at,channel_description_length,channel_hidden_subscriber
0,@Shiro-danji,1,1,567,2023-03-02T08:21:51.63267Z,216,False
1,@DaffaaPR,2,0,0,2020-07-27T09:10:36.078816Z,149,False
2,@adamardiansyah5761,8,3,1582,2013-06-17T04:53:11Z,0,False
3,@boys.252,8,3,38,2022-11-09T10:24:33.17254Z,0,False
4,@OnlyRiff,291,2,144,2021-02-26T16:08:45.913963Z,22,False
6,@raysahetapy6474,0,0,0,2021-09-20T10:38:57.996369Z,26,False
7,@Awn98,0,3,19,2016-10-31T12:48:34Z,0,False
8,@HalimahElin-ob4lf,0,0,0,2024-06-22T10:33:27.069009Z,0,False
9,@dreammashup2214,97,157,26230,2016-01-16T08:11:44Z,0,False
10,@INSol-T13,71,25,14998,2023-06-23T04:29:35.06088Z,26,False


In [16]:
# === VIDEO STATS ===
print("Video Statistics:")
display(stats_df)

Video Statistics:


,video_id,video_title,video_channel_title,source_channel,video_comment_count_api,comments_scraped,status
0,c6C8RgNb-6c,Hero Spotlight | Kimmy | Hoverjet Outrider | M...,Mobile Legends: Bang Bang Indonesia,ML,95,95,success
1,b7pLYP7ihhg,MLBB × NARUTO | Pratinjau Konten Kustom | Mobi...,Mobile Legends: Bang Bang Indonesia,ML,88,88,success
2,29FqHi9rkDQ,🔴 LIVE | MPL ID S15 | Regular Season Hari 3 Mi...,Mobile Legends: Bang Bang Indonesia,ML,85,85,success
3,T53xSsP35yk,🔴 LIVE | MPL ID S15 | Regular Season Hari 1 Mi...,Mobile Legends: Bang Bang Indonesia,ML,44,44,success
4,IMmeX4FuUQM,🔴LIVE | MDL ID S11 | Progressive Round | Hari ...,Mobile Legends: Bang Bang Indonesia,ML,15,15,success
...,...,...,...,...,...,...,...
101,vF63kGOJaAo,SUPER DUPER BIG MATCH RRQ HOSHI VS ALTER EGO M...,XINNN,XIN,68,68,success
102,Wm9uw_KmzEo,RRQ HOSHI VS ALTER EGO !!! SERU BANGET PERTAND...,XINNN,XIN,63,63,success
103,tCIWC4eMugs,2 0 SAJA BOSSS !!! EVOS HOLY VS HOME BOIS MATC...,XINNN,XIN,57,57,success
104,_u69H8dC_M0,GACOR RETRI ALBERTTT BISA BUAT COMEBACK !!! EV...,XINNN,XIN,76,76,success
